# Disaster Affected Region Tracker Analysis – P1

**Goal:** Clean disaster data with Pandas and answer the five dashboard problem statements.

The supplied CSVs do not contain latitude/longitude or GIS geometries, so this notebook produces region-level analytical visualizations rather than true GIS maps.

## 1. Upload the three CSV files

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

regions = pd.read_csv("regions.csv")
events = pd.read_csv("disaster_events.csv")
impacts = pd.read_csv("impact_assessment.csv")

print("Regions:", regions.shape)
print("Events:", events.shape)
print("Impacts:", impacts.shape)

## 2. ETL / Data Cleaning

Business rules: missing disaster types → `Unknown`; invalid dates → `NaT`; missing population → median; missing affected people/losses → `0`; duplicate records removed.

In [ ]:
regions = regions.drop_duplicates(subset=["region_id"], keep="first").copy()
events = events.drop_duplicates(subset=["event_id"], keep="first").copy()
impacts = impacts.drop_duplicates(subset=["impact_id"], keep="first").copy()

events["disaster_type"] = events["disaster_type"].fillna("Unknown")
events["event_date"] = pd.to_datetime(events["event_date"], errors="coerce")

regions["population"] = pd.to_numeric(regions["population"], errors="coerce")
population_median = regions["population"].median()
regions["population"] = regions["population"].fillna(population_median)

impacts["affected_people"] = pd.to_numeric(impacts["affected_people"], errors="coerce").fillna(0)
impacts["economic_loss_musd"] = pd.to_numeric(impacts["economic_loss_musd"], errors="coerce").fillna(0)

print("Population median used:", population_median)

## 3. Create the disaster fact dataset

In [ ]:
fact = events.merge(impacts[["event_id", "affected_people", "economic_loss_musd"]], on="event_id", how="left")
fact["affected_people"] = fact["affected_people"].fillna(0)
fact["economic_loss_musd"] = fact["economic_loss_musd"].fillna(0)
display(fact.head())

## Problem Statement 1 — Top 5 Regions by Total Affected Population

In [ ]:
top5 = (fact.groupby("region", as_index=False)["affected_people"].sum().sort_values("affected_people", ascending=False).head(5).sort_values("affected_people"))
display(top5)
plt.figure(figsize=(10,6)); plt.barh(top5["region"], top5["affected_people"]); plt.title("Top 5 Regions by Total Affected Population"); plt.xlabel("Total Affected People"); plt.ylabel("Region"); plt.tight_layout(); plt.show()

## Problem Statement 2 — Disaster Severity Distribution by Disaster Type

In [ ]:
severity_distribution = pd.crosstab(fact["disaster_type"], fact["severity"])
display(severity_distribution)
severity_distribution.plot(kind="bar", figsize=(10,6)); plt.title("Disaster Severity Distribution by Disaster Type"); plt.xlabel("Disaster Type"); plt.ylabel("Number of Events"); plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## Problem Statement 3 — Monthly Disaster Trend

In [ ]:
monthly = fact.dropna(subset=["event_date"]).copy()
monthly["month"] = monthly["event_date"].dt.to_period("M").astype(str)
monthly = monthly.groupby("month").size().reset_index(name="disaster_count")
display(monthly)
plt.figure(figsize=(12,6)); plt.plot(monthly["month"], monthly["disaster_count"], marker="o"); plt.title("Monthly Disaster Trend"); plt.xlabel("Month"); plt.ylabel("Number of Disasters"); plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

## Problem Statement 4 — Economic Loss vs Affected Population

In [ ]:
plt.figure(figsize=(10,6))
for disaster_type, group in fact.groupby("disaster_type"):
    plt.scatter(group["affected_people"], group["economic_loss_musd"], alpha=0.65, label=disaster_type)
plt.title("Economic Loss vs Affected Population"); plt.xlabel("Affected People"); plt.ylabel("Economic Loss (USD million)"); plt.legend(title="Disaster Type"); plt.tight_layout(); plt.show()

## Problem Statement 5 — Region-wise Disaster Frequency Heatmap

In [ ]:
frequency = pd.crosstab(fact["region"], fact["disaster_type"])
display(frequency)
plt.figure(figsize=(10,6)); plt.imshow(frequency.values, aspect="auto"); plt.title("Region-wise Disaster Frequency Heatmap"); plt.xlabel("Disaster Type"); plt.ylabel("Region"); plt.xticks(np.arange(len(frequency.columns)), frequency.columns, rotation=30, ha="right"); plt.yticks(np.arange(len(frequency.index)), frequency.index)
for r in range(frequency.shape[0]):
    for c in range(frequency.shape[1]): plt.text(c, r, int(frequency.iloc[r,c]), ha="center", va="center")
plt.colorbar(label="Disaster Count"); plt.tight_layout(); plt.show()

## 6. Export cleaned datasets

In [ ]:
regions.to_csv("regions_clean.csv", index=False)
events.to_csv("disaster_events_clean.csv", index=False, date_format="%Y-%m-%d")
impacts.to_csv("impact_assessment_clean.csv", index=False)
fact.to_csv("disaster_fact_clean.csv", index=False)
print("Clean datasets exported.")